# BirdCLEF+ 2026 — Baseline

**Pipeline :**
1. Config & imports
2. Exploration des données
3. Dataset (audio → mel spectrogram)
4. Modèle (EfficientNetV2-S)
5. Entraînement avec mixed precision
6. Validation (cmAP)
7. Inférence & soumission

## 1. Config & Imports

In [ ]:
import os, gc, math, time, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibilité ──────────────────────────────────────────────────────────
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
seed_everything(SEED)

# ── Chemins ───────────────────────────────────────────────────────────────────
BASE_DIR              = Path('/kaggle/input/birdclef-2026')
TRAIN_AUDIO_DIR       = BASE_DIR / 'train_audio'
TRAIN_SOUNDSCAPES_DIR = BASE_DIR / 'train_soundscapes'
TEST_SOUNDSCAPES_DIR  = BASE_DIR / 'test_soundscapes'
TRAIN_CSV             = BASE_DIR / 'train.csv'
TAXONOMY_CSV          = BASE_DIR / 'taxonomy.csv'
SOUNDSCAPES_LABELS_CSV= BASE_DIR / 'train_soundscapes_labels.csv'
SAMPLE_SUB            = BASE_DIR / 'sample_submission.csv'
OUT_DIR               = Path('/kaggle/working')

# ── Hyperparamètres ───────────────────────────────────────────────────────────
class CFG:
    # Audio
    SR          = 32_000   # sample rate
    DURATION    = 5        # secondes par chunk
    N_MELS      = 128
    N_FFT       = 1024
    HOP_LENGTH  = 320      # → ~100 frames/s
    FMIN        = 50
    FMAX        = 14_000

    # Modèle
    MODEL_NAME  = 'tf_efficientnetv2_s'
    PRETRAINED  = True
    IN_CHANS    = 1        # spectrogramme mono

    # Entraînement
    EPOCHS      = 30
    BATCH_SIZE  = 32
    LR          = 1e-3
    WEIGHT_DECAY= 1e-4
    NUM_WORKERS = 2
    N_FOLDS     = 5
    TRAIN_FOLD  = 0        # fold utilisé pour cet entraînement

    # Augmentations
    MIXUP_ALPHA = 0.5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch : {torch.__version__}')

## 2. Exploration des données

In [ ]:
# ── train.csv ────────────────────────────────────────────────────────────────
df = pd.read_csv(TRAIN_CSV)
print(f'train.csv — shape : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')
df.head()

In [ ]:
# ── taxonomy.csv → liste officielle des classes ───────────────────────────────
taxonomy  = pd.read_csv(TAXONOMY_CSV)
print(f'taxonomy.csv — shape : {taxonomy.shape}')
print(f'Colonnes : {taxonomy.columns.tolist()}')
taxonomy.head()


In [ ]:
# ── Classes issues de la taxonomy (source officielle) ─────────────────────────
# La colonne des codes espèce est 'species_code' dans BirdCLEF
CLASSES     = sorted(taxonomy['species_code'].unique().tolist())
NUM_CLASSES = len(CLASSES)
CLS2IDX     = {c: i for i, c in enumerate(CLASSES)}
print(f'Nombre de classes : {NUM_CLASSES}')

# ── train_soundscapes_labels.csv ──────────────────────────────────────────────
sc_labels = pd.read_csv(SOUNDSCAPES_LABELS_CSV)
print(f'\ntrain_soundscapes_labels.csv — shape : {sc_labels.shape}')
print(f'Colonnes : {sc_labels.columns.tolist()}')
sc_labels.head()


## 3. Dataset

In [ ]:
def load_audio(path, sr=CFG.SR, duration=CFG.DURATION):
    """Charge un fichier audio et retourne un chunk de longueur fixe."""
    target_len = sr * duration
    wav, _ = librosa.load(path, sr=sr, mono=True)
    # Sélection aléatoire d'un chunk (train) ou début (inférence)
    if len(wav) < target_len:
        wav = np.pad(wav, (0, target_len - len(wav)))
    else:
        start = random.randint(0, len(wav) - target_len)
        wav = wav[start:start + target_len]
    return wav

def audio_to_melspec(wav, sr=CFG.SR):
    """Convertit un signal audio en mel spectrogram normalisé (dB)."""
    mel = librosa.feature.melspectrogram(
        y=wav, sr=sr,
        n_mels=CFG.N_MELS,
        n_fft=CFG.N_FFT,
        hop_length=CFG.HOP_LENGTH,
        fmin=CFG.FMIN,
        fmax=CFG.FMAX,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    # Normalisation [0, 1]
    mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
    return mel.astype(np.float32)  # (N_MELS, T)

def spec_augment(mel, num_freq_masks=2, num_time_masks=2, freq_width=15, time_width=25):
    """SpecAugment : masquage fréquentiel et temporel."""
    mel = mel.copy()
    freq_max, time_max = mel.shape
    for _ in range(num_freq_masks):
        f = random.randint(0, freq_width)
        f0 = random.randint(0, freq_max - f)
        mel[f0:f0+f, :] = 0
    for _ in range(num_time_masks):
        t = random.randint(0, time_width)
        t0 = random.randint(0, time_max - t)
        mel[:, t0:t0+t] = 0
    return mel

In [ ]:
class BirdDataset(Dataset):
    """Dataset pour les fichiers train_audio (clips courts par espèce)."""
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        # train.csv : colonne 'filename' contient le chemin relatif ex: 'abcdef/XC12345.ogg'
        path = TRAIN_AUDIO_DIR / row['filename']

        # Cible one-hot (primary + secondary labels)
        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        if row['primary_label'] in CLS2IDX:
            target[CLS2IDX[row['primary_label']]] = 1.0
        if pd.notna(row.get('secondary_labels', np.nan)):
            for lbl in str(row['secondary_labels']).split():
                if lbl in CLS2IDX:
                    target[CLS2IDX[lbl]] = 0.5  # label secondaire pondéré

        wav = load_audio(path)
        mel = audio_to_melspec(wav)

        if self.augment:
            mel = spec_augment(mel)

        # (1, N_MELS, T) — channel unique
        mel = torch.tensor(mel).unsqueeze(0)
        return mel, torch.tensor(target)


class SoundscapeDataset(Dataset):
    """Dataset pour les train_soundscapes avec leurs labels (validation réaliste)."""
    def __init__(self, sc_labels_df):
        self.items = []
        step = CFG.SR * CFG.DURATION

        for _, row in sc_labels_df.iterrows():
            fp    = TRAIN_SOUNDSCAPES_DIR / row['filename']
            wav, _= librosa.load(fp, sr=CFG.SR, mono=True)
            # Le label correspond à un segment temporel précis
            start = int(row.get('start_time', 0) * CFG.SR)
            chunk = wav[start:start + step]
            if len(chunk) < step:
                chunk = np.pad(chunk, (0, step - len(chunk)))

            target = np.zeros(NUM_CLASSES, dtype=np.float32)
            for lbl in str(row.get('labels', '')).split():
                if lbl in CLS2IDX:
                    target[CLS2IDX[lbl]] = 1.0

            self.items.append((chunk, target))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        wav, target = self.items[idx]
        mel = audio_to_melspec(wav)
        mel = torch.tensor(mel).unsqueeze(0)
        return mel, torch.tensor(target)


def mixup_data(x, y, alpha=CFG.MIXUP_ALPHA):
    """Mixup augmentation."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    bs  = x.size(0)
    idx = torch.randperm(bs, device=x.device)
    return lam * x + (1 - lam) * x[idx], lam * y + (1 - lam) * y[idx]


## 4. Modèle

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, num_classes, model_name=CFG.MODEL_NAME, pretrained=CFG.PRETRAINED):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            in_chans=CFG.IN_CHANS,
            num_classes=0,     # on retire la tête
            global_pool='avg',
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)   # logits (pas de sigmoid ici)

# Vérification rapide
model = BirdModel(NUM_CLASSES).to(DEVICE)
dummy = torch.randn(2, 1, CFG.N_MELS, int(CFG.SR * CFG.DURATION / CFG.HOP_LENGTH) + 1).to(DEVICE)
print(f'Output shape : {model(dummy).shape}')  # (2, NUM_CLASSES)

## 5. Entraînement

In [ ]:
def class_mean_average_precision(y_true, y_pred):
    """cmAP — métrique officielle BirdCLEF."""
    aps = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() > 0:
            aps.append(average_precision_score(y_true[:, i], y_pred[:, i]))
    return np.mean(aps) if aps else 0.0

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Train', leave=False):
        x, y = batch
        x, y = x.to(DEVICE), y.to(DEVICE)
        x, y = mixup_data(x, y)

        optimizer.zero_grad()
        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc='Valid', leave=False):
        x, y = batch
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)

        total_loss  += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(y.cpu().numpy())

    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)
    cmap    = class_mean_average_precision(targets, preds)
    return total_loss / len(loader), cmap

In [ ]:
# ── Split train / validation sur train.csv ────────────────────────────────────
skf  = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
for fold, (trn_idx, val_idx) in enumerate(skf.split(df, df['primary_label'])):
    if fold == CFG.TRAIN_FOLD:
        break

trn_df = df.iloc[trn_idx]
val_df = df.iloc[val_idx]
print(f'Train audio — train : {len(trn_df):,}  |  valid : {len(val_df):,}')

trn_ds = BirdDataset(trn_df, augment=True)
val_ds = BirdDataset(val_df, augment=False)

# On complète la validation avec les soundscapes labelisés (contexte réel)
sc_val_ds   = SoundscapeDataset(sc_labels)
from torch.utils.data import ConcatDataset
val_ds_full = ConcatDataset([val_ds, sc_val_ds])
print(f'Valid total (clips + soundscapes) : {len(val_ds_full):,}')

trn_loader = DataLoader(trn_ds,     batch_size=CFG.BATCH_SIZE, shuffle=True,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds_full, batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)

# ── Modèle, optimiseur, scheduler ────────────────────────────────────────────
model     = BirdModel(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()
scaler    = GradScaler()

# ── Boucle d'entraînement ─────────────────────────────────────────────────────
best_cmap = 0.0
history   = []

for epoch in range(1, CFG.EPOCHS + 1):
    t0       = time.time()
    trn_loss = train_one_epoch(model, trn_loader, optimizer, scaler, criterion)
    val_loss, cmap = validate(model, val_loader, criterion)
    scheduler.step()

    elapsed = time.time() - t0
    lr_now  = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch:02d}/{CFG.EPOCHS} | '
          f'Loss {trn_loss:.4f}/{val_loss:.4f} | '
          f'cmAP {cmap:.4f} | '
          f'LR {lr_now:.2e} | '
          f'{elapsed:.0f}s')

    history.append({'epoch': epoch, 'trn_loss': trn_loss,
                    'val_loss': val_loss, 'cmap': cmap})

    if cmap > best_cmap:
        best_cmap = cmap
        torch.save(model.state_dict(), OUT_DIR / 'best_model.pth')
        print(f'  → Meilleur modèle sauvegardé (cmAP={best_cmap:.4f})')

print(f'\nMeilleur cmAP validation : {best_cmap:.4f}')


## 6. Inférence & Soumission

In [ ]:
class TestDataset(Dataset):
    """
    Dataset pour les test_soundscapes (sliding window 5s).
    Les fichiers audio sont dans test_soundscapes/ (OGG).
    Le readme.txt décrit le format ; on l'ignore ici.
    """
    def __init__(self):
        self.items = []
        sr   = CFG.SR
        step = sr * CFG.DURATION

        audio_files = sorted(TEST_SOUNDSCAPES_DIR.glob('*.ogg'))
        print(f'Fichiers test trouvés : {len(audio_files)}')

        for fp in audio_files:
            wav, _ = librosa.load(fp, sr=sr, mono=True)
            stem   = fp.stem
            for i, start in enumerate(range(0, len(wav), step)):
                chunk = wav[start:start + step]
                if len(chunk) < step:
                    chunk = np.pad(chunk, (0, step - len(chunk)))
                end_s = (i + 1) * CFG.DURATION
                self.items.append((f'{stem}_{end_s}', chunk))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row_id, wav = self.items[idx]
        mel = audio_to_melspec(wav)
        mel = torch.tensor(mel).unsqueeze(0)
        return row_id, mel


@torch.no_grad()
def predict(model, loader):
    model.eval()
    row_ids, preds = [], []
    for row_id, x in tqdm(loader, desc='Inference'):
        x = x.to(DEVICE)
        with autocast():
            logits = model(x)
        preds.append(torch.sigmoid(logits).cpu().numpy())
        row_ids.extend(row_id)
    return row_ids, np.concatenate(preds)


# ── Chargement du meilleur modèle ─────────────────────────────────────────────
model.load_state_dict(torch.load(OUT_DIR / 'best_model.pth', map_location=DEVICE))

test_ds     = TestDataset()
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

row_ids, test_preds = predict(model, test_loader)

# ── Construction du fichier de soumission ─────────────────────────────────────
sample_sub = pd.read_csv(SAMPLE_SUB)
sub_df     = pd.DataFrame(test_preds, columns=CLASSES)
sub_df.insert(0, 'row_id', row_ids)

# On garde uniquement les row_ids attendus par sample_submission.csv
sub_df = sub_df[sub_df['row_id'].isin(sample_sub['row_id'])]
sub_df = sub_df.sort_values('row_id').reset_index(drop=True)

sub_df.to_csv(OUT_DIR / 'submission.csv', index=False)
print(f'Soumission : {sub_df.shape}')
sub_df.head()
